In [ ]:
!pip install -q ijson tqdm scikit-learn

In [ ]:
import gc
import re
from pathlib import Path

import ijson
import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

DATA_DIR = Path('/content/drive/.shortcut-targets-by-id/1YwiOUwtl8pCd2GD97Q_WEzwEUtSPoxFs/TwiBot-22')
assert (DATA_DIR / 'label.csv').exists(), f'label.csv not found in {DATA_DIR}'

WORK_DIR = Path('/content/twibot22_tfidf')
WORK_DIR.mkdir(parents=True, exist_ok=True)

print('DATA_DIR =', DATA_DIR)
print('WORK_DIR =', WORK_DIR)
print('Files    :', sorted(p.name for p in DATA_DIR.iterdir())[:20])

## 1. Labels + split (label_list.py equivalent)

In [ ]:
label_df = pd.read_csv(DATA_DIR / 'label.csv')
split_df = pd.read_csv(DATA_DIR / 'split.csv')

users_index_to_uid = list(label_df['id'])
uid_to_users_index = {u: i for i, u in enumerate(users_index_to_uid)}
N_USERS = len(users_index_to_uid)

# Positive class is bot so reporting matches the RoBERTa notebook.
label_all = np.array([1 if l == 'bot' else 0 for l in label_df['label']], dtype=np.int64)
np.save(WORK_DIR / 'label_list.npy', label_all)

split_map = dict(zip(split_df['id'], split_df['split']))
split_idx = {'train': [], 'val': [], 'test': []}
for uid, idx in uid_to_users_index.items():
    s = split_map.get(uid)
    if s in split_idx:
        split_idx[s].append(idx)

print('users :', N_USERS)
print('train :', len(split_idx['train']))
print('val   :', len(split_idx['val']))
print('test  :', len(split_idx['test']))

## 2. User descriptions (stream `user.json`)

In [ ]:
USER_JSON = DATA_DIR / 'user.json'
descriptions = [''] * N_USERS

with open(USER_JSON, 'rb') as f:
    for u in tqdm(ijson.items(f, 'item'), desc='user.json'):
        uid = u.get('id')
        idx = uid_to_users_index.get(uid)
        if idx is None:
            continue
        desc = u.get('description') or ''
        descriptions[idx] = desc

print('non-empty descriptions:', sum(1 for d in descriptions if d))

## 3. Per-user tweets (`user_tweets_dict.py` equivalent)

TwiBot-22 stores tweets across `tweet_0.json ... tweet_8.json`. We build a `tid -> user_idx` map from `edge.csv` (`relation == 'post'`), then stream each tweet file and bucket texts by user.

In [ ]:
MAX_TWEETS_PER_USER = 20  # keep the same cap as the RoBERTa notebook

edge_iter = pd.read_csv(DATA_DIR / 'edge.csv', chunksize=2_000_000)
tid_to_user = {}
for chunk in tqdm(edge_iter, desc='edge.csv'):
    posts = chunk[chunk['relation'] == 'post']
    for src, tgt in zip(posts['source_id'].values, posts['target_id'].values):
        uidx = uid_to_users_index.get(src)
        if uidx is None:
            continue
        if tid_to_user.get(tgt) is None:
            tid_to_user[tgt] = uidx

print('post edges kept:', len(tid_to_user))

In [ ]:
user_tweets = [[] for _ in range(N_USERS)]
tweet_files = sorted(DATA_DIR.glob('tweet_*.json'))
print('tweet files:', [p.name for p in tweet_files])

for tf in tweet_files:
    with open(tf, 'rb') as f:
        for t in tqdm(ijson.items(f, 'item'), desc=tf.name):
            tid = t.get('id')
            uidx = tid_to_user.get(tid)
            if uidx is None:
                continue
            if len(user_tweets[uidx]) >= MAX_TWEETS_PER_USER:
                continue
            text = t.get('text') or ''
            if text:
                user_tweets[uidx].append(text)

del tid_to_user
gc.collect()
covered = sum(1 for tweets in user_tweets if tweets)
print(f'users with >=1 tweet: {covered} / {N_USERS}')
np.save(WORK_DIR / 'user_tweets_dict.npy', np.array(user_tweets, dtype=object), allow_pickle=True)

## 4. TF-IDF text features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_text(text):
    text = str(text or '')
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

combined_texts = []
for desc, tweets in tqdm(zip(descriptions, user_tweets), total=N_USERS, desc='combine_text'):
    desc_text = clean_text(desc)
    tweet_text = clean_text(' '.join(tweets))
    merged = ' '.join(part for part in [desc_text, tweet_text] if part).strip()
    combined_texts.append(merged if merged else '[empty]')

train_idx = np.array(split_idx['train'])
val_idx = np.array(split_idx['val'])
test_idx = np.array(split_idx['test'])

X_train_text = [combined_texts[i] for i in train_idx]
X_val_text = [combined_texts[i] for i in val_idx]
X_test_text = [combined_texts[i] for i in test_idx]

y_train = label_all[train_idx]
y_val = label_all[val_idx]
y_test = label_all[test_idx]

print('Creating TF-IDF features...')
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', min_df=2)
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_text)
X_val_tfidf = tfidf_vectorizer.transform(X_val_text)
X_test_tfidf = tfidf_vectorizer.transform(X_test_text)
joblib.dump(tfidf_vectorizer, WORK_DIR / 'tfidf_vectorizer.joblib')

print(f'Training set: {len(X_train_text)} samples')
print(f'Validation set: {len(X_val_text)} samples')
print(f'Test set: {len(X_test_text)} samples')
print(f'TF-IDF feature shape: {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}')

## 5. Train Logistic Regression classifier

In [ ]:
from scipy.special import expit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.svm import LinearSVC

def report(score_bot, preds, labels, tag):
    print(
        f'[{tag}] ACC={accuracy_score(labels, preds):.4f} '
        f'F1={f1_score(labels, preds):.4f} '
        f'ROC={roc_auc_score(labels, score_bot):.4f} '
        f'P={precision_score(labels, preds):.4f} '
        f'R={recall_score(labels, preds):.4f}'
    )

def select_best_threshold(prob_bot, labels, threshold_grid=np.arange(0.30, 0.71, 0.05)):
    best_threshold = 0.50
    best_val_acc = -1.0
    for thr in threshold_grid:
        preds_thr = (prob_bot >= thr).astype(int)
        val_acc = accuracy_score(labels, preds_thr)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_threshold = float(thr)
    return best_threshold, best_val_acc

models = {}

print('=' * 60)
print('TRAINING: Logistic Regression with TF-IDF features')
print('=' * 60)
logreg_model = LogisticRegression(max_iter=1000, random_state=42, solver='saga')
logreg_model.fit(X_train_tfidf, y_train)

logreg_train_prob_bot = logreg_model.predict_proba(X_train_tfidf)[:, 1]
logreg_val_prob_bot = logreg_model.predict_proba(X_val_tfidf)[:, 1]
logreg_best_threshold, logreg_best_val_acc = select_best_threshold(logreg_val_prob_bot, y_val)
logreg_train_preds = (logreg_train_prob_bot >= logreg_best_threshold).astype(int)
logreg_val_preds = (logreg_val_prob_bot >= logreg_best_threshold).astype(int)

print(f'Best validation threshold: {logreg_best_threshold:.2f}')
report(logreg_train_prob_bot, logreg_train_preds, y_train, 'train')
report(logreg_val_prob_bot, logreg_val_preds, y_val, 'val  ')

logreg_test_prob_bot = logreg_model.predict_proba(X_test_tfidf)[:, 1]
logreg_test_preds = (logreg_test_prob_bot >= logreg_best_threshold).astype(int)
print('\n=== validation-selected threshold on test ===')
report(logreg_test_prob_bot, logreg_test_preds, y_test, 'test ')
joblib.dump(logreg_model, WORK_DIR / 'tfidf_logreg.joblib')
models['logreg'] = {
    'display_name': 'Logistic Regression',
    'model': logreg_model,
    'threshold': logreg_best_threshold,
    'test_prob_bot': logreg_test_prob_bot,
    'test_preds': logreg_test_preds,
}

print('\n' + '=' * 60)
print('TRAINING: Linear SVM with TF-IDF features')
print('=' * 60)
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)

svm_train_prob_bot = expit(svm_model.decision_function(X_train_tfidf))
svm_val_prob_bot = expit(svm_model.decision_function(X_val_tfidf))
svm_best_threshold, svm_best_val_acc = select_best_threshold(svm_val_prob_bot, y_val)
svm_train_preds = (svm_train_prob_bot >= svm_best_threshold).astype(int)
svm_val_preds = (svm_val_prob_bot >= svm_best_threshold).astype(int)

print(f'Best validation threshold: {svm_best_threshold:.2f}')
print('Note: SVM P(bot) values are sigmoid-transformed decision scores, not calibrated probabilities.')
report(svm_train_prob_bot, svm_train_preds, y_train, 'train')
report(svm_val_prob_bot, svm_val_preds, y_val, 'val  ')

svm_test_prob_bot = expit(svm_model.decision_function(X_test_tfidf))
svm_test_preds = (svm_test_prob_bot >= svm_best_threshold).astype(int)
print('\n=== validation-selected threshold on test ===')
report(svm_test_prob_bot, svm_test_preds, y_test, 'test ')
joblib.dump(svm_model, WORK_DIR / 'tfidf_linear_svm.joblib')
models['svm'] = {
    'display_name': 'Linear SVM',
    'model': svm_model,
    'threshold': svm_best_threshold,
    'test_prob_bot': svm_test_prob_bot,
    'test_preds': svm_test_preds,
}

ACTIVE_MODEL = 'svm'  # change to 'logreg' if you want the logistic model instead
active_model = models[ACTIVE_MODEL]
active_model_name = active_model['display_name']
best_threshold = active_model['threshold']
test_prob_bot = active_model['test_prob_bot']
test_preds = active_model['test_preds']
(WORK_DIR / 'active_model.txt').write_text(f'{ACTIVE_MODEL}\n')
(WORK_DIR / 'best_threshold.txt').write_text(f'{best_threshold:.4f}\n')
print('\n' + '=' * 60)
print(f'ACTIVE MODEL FOR EVALUATION: {active_model_name}')
print(f'Using threshold: {best_threshold:.2f}')

## 6. Test statistics, confidence, and sample cases

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(f'Evaluating: {active_model_name}')
probs_all = np.column_stack([1.0 - test_prob_bot, test_prob_bot])
bot_scores = test_prob_bot
preds_all = test_preds
labels_all = y_test.copy()
confidences = np.maximum(bot_scores, 1.0 - bot_scores)

In [ ]:
acc = accuracy_score(labels_all, preds_all)
f1 = f1_score(labels_all, preds_all)
roc = roc_auc_score(labels_all, bot_scores)
prec = precision_score(labels_all, preds_all)
rec = recall_score(labels_all, preds_all)

print(f'Test set size : {len(labels_all)}')
print(f'Accuracy      : {acc:.4f}')
print(f'F1 (bot)      : {f1:.4f}')
print(f'ROC-AUC       : {roc:.4f}')
print(f'Precision     : {prec:.4f}')
print(f'Recall        : {rec:.4f}')

cm = confusion_matrix(labels_all, preds_all)
print('\nConfusion matrix (rows=true, cols=pred)')
print('              pred_human  pred_bot')
print(f'true_human    {cm[0,0]:>10d}  {cm[0,1]:>8d}')
print(f'true_bot      {cm[1,0]:>10d}  {cm[1,1]:>8d}')

print('\nClassification report')
print(classification_report(labels_all, preds_all, target_names=['human', 'bot'], digits=4))

In [ ]:
print(f'Mean   confidence : {confidences.mean():.4f}')
print(f'Median confidence : {np.median(confidences):.4f}')
print(f'Std    confidence : {confidences.std():.4f}')
print(f'Min / Max         : {confidences.min():.4f} / {confidences.max():.4f}')

correct = preds_all == labels_all
print(f'\nMean conf when correct : {confidences[correct].mean():.4f}')
print(f'Mean conf when wrong   : {confidences[~correct].mean():.4f}')

buckets = [(0.50, 0.60), (0.60, 0.70), (0.70, 0.80), (0.80, 0.90), (0.90, 0.95), (0.95, 1.01)]
print('\nconfidence bucket   count   accuracy')
for lo, hi in buckets:
    mask = (confidences >= lo) & (confidences < hi)
    if mask.sum() == 0:
        print(f'  [{lo:.2f}, {hi:.2f})        0       -')
    else:
        print(f'  [{lo:.2f}, {hi:.2f})   {mask.sum():>6d}    {correct[mask].mean():.4f}')

high = confidences >= 0.90
print(f'\nHigh-confidence (>=0.90) coverage : {high.mean():.2%}')
print(f'High-confidence accuracy          : {correct[high].mean():.4f}' if high.any() else 'High-confidence accuracy          : n/a')

In [ ]:
def show_case(pos, tag):
    uidx = test_idx[pos]
    uid = users_index_to_uid[uidx]
    true = 'bot' if labels_all[pos] == 1 else 'human'
    pred = 'bot' if preds_all[pos] == 1 else 'human'
    desc = (descriptions[uidx] or '').replace('\n', ' ')[:120]
    tw = user_tweets[uidx][0].replace('\n', ' ')[:140] if user_tweets[uidx] else '(no tweets)'
    print(f'[{tag}] uid={uid}  true={true}  pred={pred}  P(bot)={bot_scores[pos]:.3f}')
    print(f'    desc : {desc}')
    print(f'    tweet: {tw}\n')

order = np.argsort(-confidences)

print('=== Top-5 most-confident correct BOT predictions ===')
shown = 0
for pos in order:
    if shown >= 5:
        break
    if labels_all[pos] == 1 and preds_all[pos] == 1:
        show_case(pos, 'correct bot')
        shown += 1

print('=== Top-5 most-confident correct HUMAN predictions ===')
shown = 0
for pos in order:
    if shown >= 5:
        break
    if labels_all[pos] == 0 and preds_all[pos] == 0:
        show_case(pos, 'correct human')
        shown += 1

print('=== Top-5 most-confident MISCLASSIFICATIONS ===')
shown = 0
for pos in order:
    if shown >= 5:
        break
    if preds_all[pos] != labels_all[pos]:
        show_case(pos, 'wrong')
        shown += 1

print('=== 5 lowest-confidence predictions (model unsure) ===')
for pos in np.argsort(confidences)[:5]:
    show_case(pos, 'unsure')

In [ ]:
out_df = pd.DataFrame({
    'user_id': [users_index_to_uid[i] for i in test_idx],
    'true': ['bot' if y == 1 else 'human' for y in labels_all],
    'pred': ['bot' if y == 1 else 'human' for y in preds_all],
    'p_bot': bot_scores,
    'confidence': confidences,
    'correct': correct,
})
output_path = WORK_DIR / f'test_predictions_{ACTIVE_MODEL}.csv'
out_df.to_csv(output_path, index=False)
print('Saved', output_path, 'shape=', out_df.shape)
out_df.head()